# Three virtual nudges — continuous ten-parameter estimation (v8: improved regularization and sequential segments)

This version uses **three virtual forces**, updates the full ten-parameter estimate **at every MuJoCo timestep**, and optimizes **information gain only**. Asteroid drift and speed are **diagnostics**, never candidate-screening criteria or abort conditions. No asteroid keep-in constraint is added.

The estimator now commits each 0.2-second momentum interval **once** and continuously replaces an unfinished interval's estimate, avoiding repeated counting of heavily overlapping sliding-window rows. A weaker prior reduces the severe noiseless prior bias found in v7. These changes do **not** produce a calibrated covariance; consecutive intervals still share endpoints, and noisy results require independent testing.

The existing spacecraft CBF, contact watchdog, and force/torque/impulse/pulse bounds are retained. Virtual forces do not model arm contact or reaction on the servicer.


## 1. Import the new continuous estimator and locate the repository

Place this notebook in `active_inertial_id/notebooks/`. The import cell works whether your kernel opens in the notebook directory or the repository root. **Restart the Jupyter kernel after replacing the Python module.**

In [31]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

HERE = Path.cwd().expanduser().resolve()
ROOT = next((p for p in [HERE,*HERE.parents]
             if (p/'src'/'physical_oed_identification.py').is_file()
             and (p/'model'/'asteroid.obj').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook from the active_inertial_id directory or notebooks/ subfolder')
if str(ROOT) not in sys.path:
    sys.path.insert(0,str(ROOT))

from src.three_nudge_virtual_10param import (
    Virtual10Settings, ContinuousSpatialIdentifier, run_virtual_three)
import inspect
print('Repository:', ROOT)
print('Continuous module:', inspect.getfile(ContinuousSpatialIdentifier))

Repository: /Users/aelkebir/active_inertial_id
Continuous module: /Users/aelkebir/active_inertial_id/src/three_nudge_virtual_10param.py


## 2. Configure continuous estimation and information-only planning

The estimator processes one sensor sample and refreshes all ten estimates at **every simulation timestep** (typically 0.002 s). Each unfinished 0.2-s segment contributes **one provisional regression row**, replaced on every step; a completed segment is committed **once**. `record_period_s` controls plot/log spacing only.

`prior_sigma_normalized=10.0` is a **weaker prior** than the previous `0.25`, motivated by the earlier **noiseless** retrospective sensitivity study. It is not a noise-optimized setting. Validate against independently seeded `noise_std=1e-4` runs before drawing uncertainty or robustness conclusions.

No drift or velocity objective or asteroid keep-in constraint is introduced. Existing spacecraft CBF and actuation limits remain.


In [32]:
settings = Virtual10Settings(
    noise_std=0.0,        # first validate noiseless; supported also 1e-4
    n_nudges=3,
    n_candidates=64,
    impulse_min_Ns=0.20,
    impulse_max_Ns=2.0,
    force_min_N=1.0,
    force_max_N=4.5,
    max_impulse_Ns=3.0,
    max_force_N=7.5,
    max_torque_Nm=8.0,
    observation_choices_s=(30.0,),  # continuous 30s stream after each pulse
    estimator_window_s=0.20,
    design_prediction_period_s=1.0,  # OED predictor ONLY, not measurement rate
    record_period_s=0.10,           # plotting/logging ONLY
    prior_sigma_normalized=10.0,  # weaker prior; tune using noisy repeats, not truth
    rng_seed=19,
)
OUTPUT = ROOT/'results'/'three_nudge_virtual_10param_continuous_v8'
print('Output folder:',OUTPUT)
print('Measurement cadence: EACH simulation timestep; no observation subsampling')
print('Objective: information gain only; asteroid drift and speed logged, not constrained')


Output folder: /Users/aelkebir/active_inertial_id/results/three_nudge_virtual_10param_continuous_v8
Measurement cadence: EACH simulation timestep; no observation subsampling
Objective: information gain only; asteroid drift and speed logged, not constrained


## 3. Run the live streaming experiment

No MuJoCo resets occur between the three virtual nudges. Each step applies a known wrench (zero outside a pulse), records a body-origin state, and updates the ten-parameter estimator. Saved telemetry is downsampled for plotting only.

Candidate plans maximize E-optimal information gain with a secondary subspace information component from the existing `e_opt_score()` function. **There are no asteroid displacement or target-speed constraints, and no drift/time/impulse/force penalties in the objective.** Existing force/torque/impulse/duration restrictions, the servicer radial CBF, and unintended-contact aborts remain.

If an existing spacecraft CBF or contact check fails, the run aborts and writes `online_trace_partial.npz`, `partial_records.json`, and `failure.txt`. This is a runtime simulation safeguard, not an optimization-level trajectory guarantee.


In [33]:
result = run_virtual_three(ROOT, settings=settings, output_dir=OUTPUT, verbose=True)
assert len(result['records'])==3, 'Incomplete virtual-nudge experiment'
validation = result['validation']
print('Completed:',validation['completed_nudges'],'virtual nudges')
print('MuJoCo steps:',validation['mujoCo_steps_total'])
print('Sensor samples:',validation['every_timestep_sensor_samples'])
print('Online parameter updates:',validation['online_estimator_updates'])
print('Nominal MuJoCo timestep:',validation['dt_s'],'s')

NUDGE 1 (CONTINUOUS VIRTUAL): utility=1.468 | J=1.9744 N s | F=1.640 N | pulse=1.204 s | streaming observation=30 s
  Nudge 1 complete | online updates since previous nudge=15,602 | rank=10/10 | mass=500.2448 kg | CoM=[ 0.00158 -0.00369  0.00197] m | asteroid drift=0.1208 m
NUDGE 2 (CONTINUOUS VIRTUAL): utility=0.762 | J=1.9737 N s | F=3.602 N | pulse=0.548 s | streaming observation=30 s
  Nudge 2 complete | online updates since previous nudge=15,274 | rank=10/10 | mass=499.9054 kg | CoM=[ 0.00271 -0.00166  0.00359] m | asteroid drift=0.2112 m
Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/Users/aelkebir/Library/Python/3.9/lib/python/site-packages/IPython/core/interactiveshell.py", line 3550, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/var/folders/r4/mh57kp1943ndjdct51y2cknm0000gn/T/ipykernel_29723/2224951465.py", line 1, in <module>
    result = run_virtual_three(ROOT, settings=settings, output_dir=OUTPUT, verbose=True)
  File "/Users/aelkebir/active_inertial_id/src/three_nudge_virtual_10param.py", line 676, in run_virtual_three
  File "/Users/aelkebir/active_inertial_id/src/three_nudge_virtual_10param.py", line 356, in select_virtual_plan
    """One physical MuJoCo time-step with an externally prescribed target wrench."""
RuntimeError: Nudge 3: no feasible virtual plan found. Current estimated CoM drift=0.2112 m, speed=0.00401 m/s. Smallest predicted candidate drift=0.2838 m (limit=0.2400 m); smallest predicted speed=0.00401 m/s (limit=0.02500 m/s). Rejection counts: {'pulse_or_torque': 7, 'drif

## 4. Verify continuous estimation and single counting of completed segments

All ten parameters are estimated at each physics step, not just after a nudge. Consecutive provisional-window updates are **replacements**, not additional independent observations. The completed-segment counter should grow at approximately one per `estimator_window_s`. `stage=1` marks a force pulse; `stage=2` marks a free-flight observation.


In [ ]:
log = np.load(OUTPUT/'online_trace.npz')
t = log['time_s']
P = log['pi']
print('Saved plot samples:',len(t),'(intentionally downsampled)')
print('Physics steps:',int(log['total_simulation_steps']))
print('Online estimator updates:',int(log['total_online_estimator_updates']))
print('Sensor observations:',int(log['total_sensor_samples']))
print('Shape of continuous estimate array:',P.shape)

labels = ['Mass [kg]','m c_x [kg m]','m c_y [kg m]','m c_z [kg m]',
          'I_Oxx [kg m²]','I_Oyy [kg m²]','I_Ozz [kg m²]',
          'I_Oxy [kg m²]','I_Oxz [kg m²]','I_Oyz [kg m²]']
truth = np.asarray(result['true_pi_DEV_ONLY'])
fig, axes = plt.subplots(5,2,figsize=(13,15),sharex=True)
for k,ax in enumerate(axes.flat):
    ax.plot(t,P[:,k],linewidth=1.0,label='Live estimate')
    ax.axhline(truth[k],linestyle='--',linewidth=1.2,label='MuJoCo truth (post-hoc)')
    ax.set_ylabel(labels[k]); ax.grid(alpha=.25)
    if k==0: ax.legend(loc='best',fontsize=8)
for ax in axes[-1]: ax.set_xlabel('MuJoCo time [s]')
fig.suptitle('All ten parameter estimates — updated at every timestep',fontsize=14)
plt.tight_layout()
plt.show()
print('Completed non-overlapping segments:', int(log['total_completed_estimator_segments']))


## 5. Observability and conditioning

Rank and singular values are computed from the sequential interval normal matrix plus the **current** partial interval. Every completed interval contributes once; unlike v7, hundreds of overlapping sliding windows are not cumulatively counted as independent rows.

However, neighboring intervals share endpoint sensor errors. This is not a full errors-in-variables model or generalized least-squares covariance. Rank is **numerical only**, and `P` remains a regularized sensitivity proxy, not calibrated uncertainty.


In [ ]:
rank = log['numerical_rank']
sv = log['singular_values']
fig,ax = plt.subplots(1,2,figsize=(12,4))
ax[0].step(t,rank,where='post');ax[0].set(xlabel='Time [s]',ylabel='Numerical rank',ylim=(0,10.5))
ax[0].grid(alpha=.25)
for k in [0,3,6,9]:
    ax[1].semilogy(t,np.maximum(sv[:,k],1e-15),label=f'SV {k+1}')
ax[1].set(xlabel='Time [s]',ylabel='Scaled singular value (log scale)')
ax[1].grid(alpha=.25);ax[1].legend()
plt.tight_layout();plt.show()
print('Final numerical rank:',int(rank[-1]),'/10')
if rank[-1]<10:
    print('Some directions remain weak: three force schedules may be insufficient.')

In [ ]:
fig,ax=plt.subplots(figsize=(9,2.6))
ax.plot(t, log['completed_segments'], label='Committed non-overlapping intervals')
ax.set(xlabel='Simulation time [s]', ylabel='Completed segments', title='One committed interval per estimator window')
ax.grid(alpha=.25);ax.legend();plt.show()


## 6. Compare final estimates with truth (post-run validation only)

Ground truth is never used in `select_virtual_plan()` or `ContinuousSpatialIdentifier`. The inertia-parameter validation oracle runs **only after all three nudges have completed**. MuJoCo truth is also used during simulation for independent **safety watchdogs and diagnostic trajectory plots**, but never for online estimation or nudge optimization.

In [ ]:
final = result['identifier'].pi
report = pd.DataFrame({'Parameter':labels,'True (post-hoc only)':truth,
                       'Online final estimate':final,'Absolute error':abs(final-truth)})
display(report)
print('Mass relative error:',f'{100*abs(validation["mass_relative_error"]):.4f}%')
print('CoM offset error:',f'{1000*validation["com_error_m"]:.4f} mm')
print('I_O Frobenius error:',f'{validation["inertia_O_frobenius_error_kgm2"]:.5f} kg m²')

## 7. Confirm integrated impulses, runtime safety checks, and continuous nudge schedule

MuJoCo applies the known virtual force at a moving body-fixed application point. Linear impulse and moment impulse are integrated **at each timestep** about a fixed world origin. The integrated world-frame impulse magnitude need not be exactly the nominal body-frame magnitude if the asteroid rotates during the pulse.

In [ ]:
display(pd.DataFrame([{
    'Nudge':r['nudge'],
    'Commanded J [N s]':r['impulse_commanded_Ns'],
    'Integrated world |J| [N s]':r['impulse_applied_world_magnitude_Ns'],
    'Force [N]':r['force_commanded_N'],
    'Pulse time [s]':r['pulse_s'],
    'Observation time [s]':r['observation_s'],
    'Streaming estimator updates':r['online_updates_during_this_nudge_and_observation'],
    'Numerical rank at end':r['numerical_rank'],
    'Asteroid CoM drift [m]':r['drift_m'],
    'CBF interventions':r['cbf_interventions_total'],
} for r in result['records']]))
assert all(r['impulse_commanded_Ns']<=settings.max_impulse_Ns+1e-9 for r in result['records'])
assert all(r['force_commanded_N']<=settings.max_force_N+1e-9 for r in result['records'])
print('Saved files:', sorted(p.name for p in OUTPUT.iterdir()))

## 8. Visualize asteroid motion and moment accumulation

The plot includes asteroid CoM **truth for visualization only**, never used by the estimator or planner. The moment impulse is referenced to the fixed world origin, consistent with the ten-parameter momentum equation.

In [ ]:
com = log['asteroid_com_truth_diagnostic_m']
J = log['applied_linear_impulse_Ns']
L = log['applied_angular_impulse_world_origin_Nms']
fig=plt.figure(figsize=(8,5))
ax=fig.add_subplot(111,projection='3d')
ax.plot(com[:,0],com[:,1],com[:,2],lw=1.4)
ax.scatter(*com[0],s=35,label='Initial CoM')
ax.scatter(*com[-1],s=35,label='Final CoM')
ax.set(xlabel='X [m]',ylabel='Y [m]',zlabel='Z [m]',title='Asteroid CoM path')
ax.legend();plt.tight_layout();plt.show()

fig,axes=plt.subplots(2,1,figsize=(10,6),sharex=True)
for i,c in enumerate('xyz'):
    axes[0].plot(t,J[:,i],label=f'J_{c}')
    axes[1].plot(t,L[:,i],label=f'L_{c}')
axes[0].set_ylabel('Cumulative linear impulse [N s]')
axes[1].set_ylabel('Cumulative world-origin moment impulse [N m s]')
axes[1].set_xlabel('Time [s]')
for a in axes: a.legend(ncol=3);a.grid(alpha=.25)
plt.tight_layout();plt.show()

## 9. Track estimation quality throughout the online run

This is a **post-run diagnostic**, not data used by the estimator or optimizer. The plotted physical-property errors use MuJoCo truth only for validation. A full numerical rank does not imply a low estimation error, and the history reveals whether identification improves during pulses or also during free flight.


In [ ]:
from src.physical_oed_identification import com_and_ic, matrix_inertia

m_true = truth[0]
c_true, Ic_true = com_and_ic(truth)
mass_relative_error_pct = 100.0 * np.abs(P[:, 0] - m_true) / m_true
com_error_mm = 1000.0 * np.linalg.norm(P[:, 1:4] / P[:, 0, None] - c_true, axis=1)
inertia_error = np.array([np.linalg.norm(matrix_inertia(p) - matrix_inertia(truth)) for p in P])

fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
for ax, y, name in zip(axes,
                       [mass_relative_error_pct, com_error_mm, inertia_error],
                       ['Mass relative error [%]', 'CoM estimation error [mm]',
                        'Inertia-about-origin tensor error [kg m²]']):
    ax.plot(t, y)
    ax.set_ylabel(name)
    ax.grid(alpha=0.25)
    for r in result['records']:
        ax.axvline(r['end_sim_time_s'], lw=.8, linestyle=':', alpha=.5)
axes[-1].set_xlabel('MuJoCo time [s]')
fig.suptitle('True estimation error throughout the run (post-hoc only)')
plt.tight_layout()
plt.show()

fig, axs = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axs[0].plot(t, log['weighted_rate_residual'])
axs[0].set_ylabel('Whitened window residual norm')
axs[1].plot(t, log['last_window_s'])
axs[1].set_ylabel('Window elapsed time [s]')
axs[1].set_xlabel('MuJoCo time [s]')
for ax in axs:
    ax.grid(alpha=.25)
plt.tight_layout()
plt.show()
print('Note: adjacent completed segments share endpoint measurement errors; these residuals are not independent innovations.')


## 10. Regularization sensitivity: same v8 measurements

Re-solve the saved v8 normal equations for several prior strengths, **without rerunning MuJoCo**. A weaker prior can reduce the bias in ideal noiseless measurements, but in noisy data it may amplify errors. The retrospective fit is **not** evidence that sigma 10 is optimal under noise; evaluate multiple random seeds and compare mass, CoM, and inertia error only in the separate post-run validation.


In [ ]:
from src.physical_oed_identification import SCALE
G = np.asarray(log['normal_matrix'], float)
g = np.asarray(log['normal_rhs'], float)
prior = np.asarray(log['prior_pi'], float)
scale = np.asarray(log['parameter_scale'], float)
regularization_rows = []
for sigma in [0.25, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0]:
    pi_trial = prior + scale * np.linalg.solve(G + np.eye(10) / sigma**2, g)
    try:
        c_trial, _ = com_and_ic(pi_trial)
        valid = True
    except ValueError:
        c_trial = np.full(3, np.nan)
        valid = False
    regularization_rows.append({
        'Normalized prior sigma': sigma,
        'Physically admissible': valid,
        'Mass error [kg]': abs(pi_trial[0] - truth[0]),
        'CoM error [mm]': 1000*np.linalg.norm(c_trial-c_true),
        'Inertia tensor error [kg m²]': np.linalg.norm(matrix_inertia(pi_trial)-matrix_inertia(truth)),
    })
display(pd.DataFrame(regularization_rows))
print('Post-hoc diagnostic only; do not select a prior by minimizing errors against MuJoCo truth.')


## Compare this complete v8 run to the previous v7 run (optional)

This compares **post-run validation errors** if a prior v7 run was saved. Because the estimator also changes the nudge-selection trajectory, these are **whole-pipeline** outcomes, not an isolated estimator benchmark. A controlled same-data replay is needed to isolate the estimator contribution.


In [ ]:
previous = ROOT/'results'/'three_nudge_virtual_10param_continuous_info_only'/'validation_DEV_ONLY.json'
if previous.exists():
    import json
    old_result = json.loads(previous.read_text())
    def summary_row(label, val):
        return {'Run':label,
                'Mass absolute error [kg]': abs(float(val['mass_relative_error'])) * float(val['true_pi_DEV_ONLY'][0]),
                'CoM error [mm]': 1000 * float(val['com_error_m']),
                'Inertia tensor error [kg m²]': float(val['inertia_O_frobenius_error_kgm2'])}
    display(pd.DataFrame([summary_row('Previous v7',old_result),
                          summary_row('New v8', validation)]))
    print('Note: different online estimates may change selected nudges; this comparison is not a controlled causal ablation.')
else:
    print('No prior v7 validation file found. V8 results are saved separately in:', OUTPUT)


## 11. Noisy-data validation

After the zero-noise run, repeat the experiment with `noise_std=1e-4` under multiple seeds. Compare final errors and time histories against v7. Noisy pose/twist regressors contain errors in both the model matrix and response; weakening the prior does not automatically solve this. Neighboring sequential intervals also share a measured endpoint, so formal uncertainty requires a proper noise model.


In [ ]:
# Optional independent noisy run (uncomment):
# from dataclasses import replace
# noisy_settings = replace(settings, noise_std=1e-4, rng_seed=20)
# noisy_result = run_virtual_three(
#     ROOT, settings=noisy_settings,
#     output_dir=ROOT/'results'/'three_nudge_virtual_10param_continuous_noisy',
# )